# Understanding the Roman WFI Dark Current Reference File 

## Kernel Information and Read-Only Status

To run this notebook, please select "Roman Research Nexus {VERSION}" kernel at the top right of your window. For example "Roman Research Nexus 2026.1".

This notebook is read-only. You can run cells and make edits, but you must save changes to a different location. We recommend saving the notebook within your home directory, or to a new folder within your home (e.g. <span style="font-variant:small-caps;">file > save notebook as > my-nbs/nb.ipynb</span>). Note that a directory must exist before you attempt to add a notebook to it.
    

## Introduction
The purpose of this notebook is to understand the content and purpose of the **Dark Current** (`DARK`) reference file.

More details about this and other reference files can be found in the [Reference File Information](https://roman-pipeline.readthedocs.io/en/stable/roman/references_general/index.html)

### Local Run Settings

If you want to run the notebook in your local machine, refer to the information in [local installation](../../markdown/local-run.md) instructions before proceeding with the notebook. The instructions provide important information about setting up your environment and installing dependencies.

## Imports
Libraries used:
- *astropy* for image normalization
- *copy* for making copies of Python objects
- *crds* for access to calibration reference files
- *matplotlib* and *mpl_toolkits* for plotting images
- *numpy* for array manipulation
- *roman_datamodels* for opening Roman WFI ASDF files
- *asdf* for opening Roman WFI ASDF files
- *os* for operating system functions

In [1]:
import os
from astropy.visualization import simple_norm
import copy

import matplotlib.pyplot as plt
from matplotlib import colors, colormaps as cm
from mpl_toolkits.axes_grid1 import make_axes_locatable
import numpy as np
import roman_datamodels as rdm

### The Calibration Reference Data System (CRDS)

The reference files, developed and validated by STScI’s Science Operations Center, are continually updated as new WFI data become available. For more information about how CRDS works and how it assigns the most appropriate reference file for each calibration step, refer to the notebook [Understanding CRDS and How to Select Calibration Reference files](crds_reference_files.ipynb). 

**IMPORTANT NOTE:** Reference files are a work in progress and will be updated several times before Roman launch. If you notice irregularities or missing information, please understand that they may be a known issue. If you have questions, please contact the [Roman Help Desk](https://romanhelp.stsci.edu).

In [2]:
import crds

Now let's dive into this reference file type.

### Darks 

The DARK reference file is selected based on the WFI mode (imaging or spectroscopy) used to obtain the science data. During the `romancal.dark_current.DarkCurrentStep()` step, the dark current is subtracted on a pixel-by-pixel and resultant-by-resultant basis. Pixels that are undefined in the dark reference file will not be subtracted from the science data.

The dark reference files are created from dark calibration datasets.  A set of dark files are sigma clipped, and stacked resultant-by-resultant to create a super dark.

For more details, see the [romancal documentation](https://roman-pipeline.readthedocs.io/en/latest/roman/dark_current/index.html) and [Rdox documentation](https://roman-docs.stsci.edu/data-handbook-home/roman-data-pipelines/exposure-level-pipeline#ExposureLevelPipeline-dark_current) for dark current subtraction.


Before proceeding, let's check the environmental variables set for CRDS

In [3]:
print(f"CRDS server location: {os.environ.get('CRDS_SERVER_URL')}")
print(f"CRDS context file: {os.environ.get('CRDS_CONTEXT')}")

If we want to change the context, we can do it in the next cell. In this case, we choose context `roman_0055.pmap`.

In [4]:
os.environ['CRDS_CONTEXT']='roman_0055.pmap'

### Retrieving Reference Files

As you run the exposure pipeline, the most up-to-date reference files will be automatically selected for each step. However, if you would like to use a specific reference file, these can be retrieved through the `crds` Python API and the ELP run with those files (more on that later). Let's begin with how to access reference files from CRDS.

First, let's start by looking at the `crds.getrecommendations()` function. This function returns a dictionary of file names that match the criteria that you supply. Selection criteria are specified in a dictionary of key-value pairs. Each Roman WFI metadata keyword in the dictionary is all-caps and always begins with "ROMAN.META.". The remaining parts of the string correspond to the metadata keyword locations in the science data file schema. Different reference file types require different combinations of science metadata to match to the reference files. In general, all reference file types will require the instrument name ("INSTRUMENT.NAME") and start time ("EXPOSURE.START_TIME"). Most file types require the detector name ("INSTRUMENT.DETECTOR"), and some file types require the exposure type ("EXPOSURE.TYPE") or optical element ("INSTRUMENT.OPTICAL_ELEMENT").

For the dark files in particular, the required keywords are:
- dark
    - ROMAN.META.INSTRUMENT.NAME
    - ROMAN.META.INSTRUMENT.DETECTOR
    - ROMAN.META.EXPOSURE.TYPE
    - ROMAN.META.EXPOSURE.START_TIME

These keywords may be combined into a single dictionary to find multiple reference file types using `crds.getreferences()`. For example, if you would like to find the names of the dark reference files used by the pipeline, you could run the following example:

In [5]:
meta = {'ROMAN.META.INSTRUMENT.NAME': 'WFI',
        'ROMAN.META.INSTRUMENT.DETECTOR': 'WFI01',
        'ROMAN.META.INSTRUMENT.OPTICAL_ELEMENT': 'F158',
        'ROMAN.META.EXPOSURE.TYPE': 'WFI_IMAGE',
        'ROMAN.META.EXPOSURE.START_TIME': '2026-01-01 00:00:00'
       }

ref_files = crds.getrecommendations(meta, reftypes=['dark'], observatory='roman')

The `ref_files` variable now contains a dictionary for each of the `DARK` reference file you requested. These are the reference files that correspond to a science observation taken at midnight UTC on January 1, 2026 in the WFI imaging mode with optical element F158 and detector WFI01. Let's take a look at the names of the files CRDS returned:

In [6]:
ref_files

We can also use `crds.getreferences()` to accomplish the same thing; however, `getreferences()` goes one step further beyond `getrecommendations()` and will download the reference files if they are not already in your local cache. Using the same example as above:

In [7]:
meta = {'ROMAN.META.INSTRUMENT.NAME': 'WFI',
        'ROMAN.META.INSTRUMENT.DETECTOR': 'WFI01',
        'ROMAN.META.EXPOSURE.TYPE': 'WFI_IMAGE',
        'ROMAN.META.EXPOSURE.START_TIME': '2026-01-01 00:00:00'
       }

ref_files = crds.getreferences(meta, reftypes=['dark'], observatory='roman')

And once again we can examine the output of `ref_files`:

In [8]:
ref_files

### Examining Reference Files

Reference files use `roman_datamodels` just like WFI science data products and can be accessed in the same way (see the tutorial [Working with ASDF](../working_with_asdf/working_with_asdf.ipynb) for more information). Let's take a closer look at the files we retrieved from our `crds.getreferences()` example:

In [9]:
dark = rdm.open(ref_files['dark'])
dark.info()

We see that the dark reference file contains metadata plus several arrays:
- The `data` array contains a cube of dark values up-the-ramp. This is used in the current version of the Exposure Pipeline, however changes are being made towards a dark rate subtraction post-ramp-fitting. 
- The `dark_slope` array contains the dark rate per pixel
- The `dark_slope_error` contains the uncertainty in the dark rate. 
- The `dq` array is present to flag effects in the dark current (such as hot pixels).

Let's take a look at the dark for this detector. First, lets check the shape of the data array:

In [10]:
print("dark.data shape:", dark.data.shape)
print("dark.dq shape:", dark.dq.shape)
print("dark.dark_slope shape:", getattr(dark, 'dark_slope', None).shape if hasattr(dark, 'dark_slope') else "No dark_slope")

Now lets get some basic statistics on the cube (or a representative slice)

In [1]:
# Example: inspect one of the coefficient arrays (adjust name after checking .info())
if hasattr(refpix, 'alpha') and refpix.alpha is not None:
    coeff = refpix.alpha
    print("\nStatistics for alpha coefficients:")
    print("  Min:", coeff.min(), "Max:", coeff.max(),
          "Mean:", coeff.mean(), "Std:", coeff.std())

NameError: name 'refpix' is not defined

Note that in this case the pre-launch darks are all zero, so the plot will be rather flat.

Let's get a quick histogram of a slice:

In [12]:
plt.figure(figsize=(8,5))
plt.hist(data_slice.flatten(), bins=100, range=(data_slice.min(), np.percentile(data_slice, 99.5)), log=True)
plt.xlabel('Dark Current Value')
plt.ylabel('Pixel Count (log)')
plt.title('Pixel Value Distribution (slice)')
plt.show()

Note: In flight data the dark current will be non-zero and these visualizations
will show clear structure (hot pixels, gradients, etc.).

Now let's check the dark image

In [13]:
fig, axs = plt.subplots(1, 3, figsize=(18, 6))  # Add a third panel for dark_slope if available

slice_idx = 10  # You can change this index to visualize a different slice of the dark current data
data_slice = dark.data[slice_idx]

my_cmap = copy.copy(cm.get_cmap('nipy_spectral'))
my_cmap.set_bad('black')

norm = simple_norm(data_slice, stretch='sqrt', percent=99.5, min_percent=0.5)

im0 = axs[0].imshow(data_slice, cmap=my_cmap, norm=norm, origin='lower')
divider = make_axes_locatable(axs[0])
cax = divider.append_axes("right", size="5%", pad=0.05)
fig.colorbar(im0, cax=cax, label='DN/s or electrons')
axs[0].set_title(f'Dark Current (slice {slice_idx})')

# DQ panel
axs[1].imshow(np.bool_(dark.dq), cmap='binary_r', origin='lower')
axs[1].set_title('Dark Current DQ Flags')

# dark_slope (rate image) 
if hasattr(dark, 'dark_slope') and dark.dark_slope is not None:
    slope_norm = simple_norm(dark.dark_slope, stretch='sqrt', percent=99)
    im2 = axs[2].imshow(dark.dark_slope, cmap=my_cmap, norm=slope_norm, origin='lower')
    divider2 = make_axes_locatable(axs[2])
    cax2 = divider2.append_axes("right", size="5%", pad=0.05)
    fig.colorbar(im2, cax=cax2, label='DN/s or electrons')
    axs[2].set_title('Dark Slope (Rate)')

for ax in axs:
    ax.set_xlabel('Science X (pixels)')
    ax.set_ylabel('Science Y (pixels)')

plt.tight_layout()
plt.show()

## About this Notebook
**Author:** T. Desjardins & R. Diaz

**Updated On:** 2026-07-06

<table width="100%" style="border:none; border-collapse:collapse;">

  <tr style="border:none;">
    <td style="border:none; width:180px; white-space:nowrap;">
       <a href="#top" style="text-decoration:none; color:#0066cc;"> Top of page</a>
    </td>
    <td style="border:none; text-align:center;">
        <img src="https://raw.githubusercontent.com/spacetelescope/roman_notebooks/refs/heads/main/roman_logo.png" alt="roman_logo" width="50px">
    </td>
    <td style="border:none; text-align:right;">
       <img src="https://raw.githubusercontent.com/spacetelescope/roman_notebooks/refs/heads/main/stsci_logo2.png" width="90">
    </td>
  </tr>
</table>